 Scenario: Automated Learning Hub for an EdTech Company
🎓 Problem Statement
An EdTech company wants to:
- Keep track of the latest educational technology innovations
- Automatically generate weekly blog posts for their website
- Reduce time spent on manual research and content drafting
👉 So we build a multi-agent AI system using CrewAI to automate this workflow:
- Trend Monitoring Agent → Continuously scans for new EdTech tools, teaching strategies, and AI-in-education updates
- Content Summarization Agent → Condenses research into digestible insights tailored for educators and learners
- Blog Writing Agent → Crafts engaging, SEO-friendly blog posts automatically
- Publishing Agent → Schedules and posts content directly to the company’s website

In [6]:
import os
from IPython.display import display, HTML
from google.colab import userdata
from crewai import Agent, Task, Crew, Process

# =========================================================
# 1. ENV SETUP
# =========================================================
os.environ["CREWAI_TRACING_ENABLED"] = "true"

raw_key = userdata.get("GROQ_API_KEY")
clean_key = str(raw_key).strip().replace("\n", "").replace("\r", "")

if not clean_key:
    raise ValueError("GROQ_API_KEY not found. Please add it in Colab userdata.")

os.environ["GROQ_API_KEY"] = clean_key

# remove OPENAI key if present
os.environ.pop("OPENAI_API_KEY", None)

print("✅ API Key Loaded Safely")

# =========================================================
# 2. USER INPUT
# =========================================================
user_topic = input("Enter the EdTech focus topic: ").strip()
if not user_topic:
    raise ValueError("Topic cannot be empty.")

target_audience = input(
    "Enter the target audience (e.g. educators, students, schools, parents): "
).strip()

if not target_audience:
    target_audience = "educators and learners"

# =========================================================
# 3. LLM
# =========================================================
llm = "groq/llama-3.3-70b-versatile"

# =========================================================
# 4. AGENTS
# =========================================================
trend_monitor = Agent(
    role="Trend Monitoring Agent",
    goal="Identify recent and relevant EdTech innovations, AI-in-education trends, digital learning tools, and teaching strategies.",
    backstory=(
        "You are an expert EdTech research analyst. "
        "You track educational innovation, digital pedagogy, LMS evolution, AI tutors, "
        "personalized learning systems, and classroom technology."
    ),
    llm=llm,
    verbose=True
)

summarizer = Agent(
    role="Content Summarization Agent",
    goal="Convert raw EdTech findings into digestible, meaningful, and audience-specific insights.",
    backstory=(
        "You are a strategic content analyst who separates signal from noise, "
        "highlights educational relevance, and explains why a trend matters in practice."
    ),
    llm=llm,
    verbose=True
)

blog_writer = Agent(
    role="Blog Writing Agent",
    goal="Write professional, engaging, SEO-friendly EdTech blog posts.",
    backstory=(
        "You are a senior blog strategist who writes educational content that is clear, "
        "structured, modern, and valuable for readers."
    ),
    llm=llm,
    verbose=True
)

publisher = Agent(
    role="Publishing Agent",
    goal="Prepare final content for website publication with metadata, publishing plan, and structured presentation.",
    backstory=(
        "You are a digital publishing specialist. You optimize content packaging, "
        "SEO structure, readability, and release scheduling for websites."
    ),
    llm=llm,
    verbose=True
)

# =========================================================
# 5. TASKS
# =========================================================
trend_task = Task(
    description=f"""
Research the topic: "{user_topic}"

Focus areas:
- Latest EdTech innovations
- AI in education
- Smart classroom tools
- Digital pedagogy trends
- Adaptive learning systems
- LMS/platform innovations
- Gamified learning models

Audience context: {target_audience}

Deliverables:
1. Identify at least 5 major trends or innovations
2. Explain each trend in a practical way
3. Mention why it matters for the target audience
4. Keep the output structured and professional
""",
    expected_output="""
A structured list of 5-7 EdTech trends with:
- trend name
- short explanation
- why it matters
- likely impact
""",
    agent=trend_monitor
)

summary_task = Task(
    description=f"""
Using the research from the Trend Monitoring Agent, create a deeper synthesis.

Requirements:
- Condense the findings into practical strategic insights
- Highlight educational value, implementation relevance, and audience impact
- Add depth instead of repeating research
- Organize insights clearly for blog creation and system explanation

Audience: {target_audience}

Also include:
- key takeaways
- opportunities
- risks/challenges
- future outlook
""",
    expected_output="""
A refined summary with:
- strategic insights
- practical implications
- opportunities
- risks
- future direction
""",
    agent=summarizer,
    context=[trend_task]
)

blog_task = Task(
    description=f"""
Using the summarized insights, write a complete blog post on:
"{user_topic}"

Requirements:
- Strong SEO-friendly title
- Engaging introduction
- Main body with headings and subheadings
- Explain major trends in a reader-friendly way
- Include practical recommendations
- Add conclusion
- Add a soft call-to-action
- Style should be professional and suitable for an EdTech company website

Also include a separate section titled:
"System Design Explanation"

In that section explain the multi-agent workflow:
- Trend Monitoring Agent
- Content Summarization Agent
- Blog Writing Agent
- Publishing Agent

Also generate these in ASCII / diagrammatic format:
1. Architecture diagram
2. Workflow pipeline diagram
3. Agent interaction flow

Make the output visually structured and presentation-ready.
""",
    expected_output="""
A full blog post plus:
- project/system title
- system explanation
- ASCII architecture diagram
- workflow diagram
- agent interaction flow
""",
    agent=blog_writer,
    context=[summary_task]
)

publishing_task = Task(
    description=f"""
Take the final blog and package it for publication.

Generate:
1. Final polished article
2. SEO title
3. Meta description
4. Suggested slug
5. Suggested tags
6. Suggested weekly publishing schedule
7. Final clean ASCII diagrams
8. A final section called "Deployment & Scaling Notes"

The final output must be highly structured and include:
- Project Title
- Scenario Overview
- Business Problem
- Why Automation is Needed
- Agent Roles in Depth
- End-to-End Workflow
- Inputs and Outputs per Stage
- Benefits
- Risks / Limitations
- Future Scope
- ASCII Diagrams

Keep the formatting neat so it can be copied into a report, PPT, or documentation.
""",
    expected_output="""
A publication-ready final document with:
- polished blog
- SEO assets
- scheduling notes
- architecture explanation
- ASCII diagrams
- deployment/scaling notes
""",
    agent=publisher,
    context=[blog_task]
)

# =========================================================
# 6. CREW
# =========================================================
crew = Crew(
    agents=[trend_monitor, summarizer, blog_writer, publisher],
    tasks=[trend_task, summary_task, blog_task, publishing_task],
    process=Process.sequential,
    verbose=True
)

# =========================================================
# 7. RUN
# =========================================================
result = crew.kickoff()
final_output = str(result)

print("\n✅ FINAL OUTPUT:\n")
print(final_output)

# =========================================================
# 8. HTML RENDER
# =========================================================
safe_output = (
    final_output.replace("&", "&amp;")
    .replace("<", "&lt;")
    .replace(">", "&gt;")
)

html_output = f"""
<div style="
    max-width: 1100px;
    margin: 20px auto;
    padding: 28px;
    border-radius: 18px;
    background: #f8fafc;
    border: 1px solid #dbe3ea;
    font-family: Arial, sans-serif;
    color: #1f2937;
    line-height: 1.7;
">
    <h1 style="color:#0f172a; margin-bottom:8px;">
        Automated Learning Hub for an EdTech Company
    </h1>

    <p style="font-size:16px; margin:0 0 8px 0;">
        <strong>Topic:</strong> {user_topic}
    </p>

    <p style="font-size:16px; margin:0 0 18px 0;">
        <strong>Audience:</strong> {target_audience}
    </p>

    <div style="
        background:#ffffff;
        border:1px solid #e5e7eb;
        border-radius:14px;
        padding:20px;
        white-space:pre-wrap;
        font-size:15px;
        overflow-x:auto;
    ">{safe_output}</div>
</div>
"""

display(HTML(html_output))

✅ API Key Loaded Safely
Enter the EdTech focus topic: Design a complete Automated Learning Hub system for an EdTech company using a multi-agent AI architecture.  Context: An EdTech company wants to automate how it discovers, analyzes, writes, and publishes educational content.  Objectives: - Track latest EdTech innovations, AI in education, and digital learning trends - Automatically generate high-quality weekly blog posts - Reduce manual effort in research, summarization, and writing - Ensure SEO optimization and consistent publishing  Agents in the system: 1. Trend Monitoring Agent → finds latest EdTech tools, AI learning platforms, LMS trends, and teaching innovations 2. Content Summarization Agent → converts research into practical insights for educators and learners 3. Blog Writing Agent → creates structured, engaging, SEO-friendly blog posts 4. Publishing Agent → prepares content for website publishing with SEO metadata and scheduling  Task: Generate a complete system-level expla

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 91f3bd03-151c-4ec1-ae5f-8ba358eea4fa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Research the topic: "Design a complete Automated Learning Hub system for an EdTech company using a             │
│  multi-agent AI architecture.  Context: An EdTech company wants to automate how it discovers, analyzes,         │
│  writes, and publishes educational content.  Objectives: - Track latest EdTech innovations, AI in education,    │
│  and digital learning trends - Automatically generate high-quality weekly blog posts - Reduce manual effort in  │
│  research, summarization, and writing - Ensure SEO optimization and consistent publishing  Agents in the        │
│  system: 1. Trend Monitoring Agent → finds latest EdTech tools, AI learning platforms, LMS trends, and          │
│  teaching innovations 2. Content Summarization Agent → converts research into practical insights for educators  │
│  and learners 3. Blog Writing Agent → creates structured, engaging, SEO-friendly blog posts 4. Publishing       │
│  Agent → prepares content for website publishing with SEO metadata and scheduling  Task: Generate a complete    │
│  system-level explanation and blog output.  Output Requirements: 1. Project Title 2. Scenario Overview 3.       │
│  Business Problem 4. Why Automation is Needed 5. Detailed Explanation of Each Agent (role, input, output,       │
│  responsibility) 6. End-to-End Workflow (step-by-step) 7. Input → Output mapping for each stage 8. Benefits of  │
│  this architecture 9. Risks / limitations 10. Future improvements and scalability  VERY IMPORTANT: Also         │
│  generate the following in diagram format:  1. Architecture Diagram (text/ASCII format) 2. Workflow Pipeline    │
│  Diagram (with arrows) 3. Agent Interaction Flow  Formatting: - Make the output visually structured and         │
│  presentation-ready - Use boxes, arrows, sections, and spacing - Keep it suitable for a final-year B.Tech       │
│  project report - Make it deep, not generic  Also include: - A full SEO-optimized blog post based on the        │
│  system - SEO title, meta description, tags, and slug - Publishing schedule suggestion  Tone: Professional,     │
│  structured, and technical with real-world clarity."                                                            │
│                                                                                                                 │
│  Focus areas:                                                                                                   │
│  - Latest EdTech innovations                                                                                    │
│  - AI in education                                                                                              │
│  - Smart classroom tools                                                                                        │
│  - Digital pedagogy trends                                                                                      │
│  - Adaptive learning systems                                                                                    │
│  - LMS/platform innovations                                                                                     │
│  - Gamified learning models                                                                                     │
│                                                                                                                 │
│  Audience context: schools                                                                                      │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Monitoring Agent                                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Research the topic: "Design a complete Automated Learning Hub system for an EdTech company using a             │
│  multi-agent AI architecture.  Context: An EdTech company wants to automate how it discovers, analyzes,         │
│  writes, and publishes educational content.  Objectives: - Track latest EdTech innovations, AI in education,    │
│  and digital learning trends - Automatically generate high-quality weekly blog posts - Reduce manual effort in  │
│  research, summarization, and writing - Ensure SEO optimization and consistent publishing  Agents in the        │
│  system: 1. Trend Monitoring Agent → finds latest EdTech tools, AI learning platforms, LMS trends, and          │
│  teaching innovations 2. Content Summarization Agent → converts research into practical insights for educators  │
│  and learners 3. Blog Writing Agent → creates structured, engaging, SEO-friendly blog posts 4. Publishing       │
│  Agent → prepares content for website publishing with SEO metadata and scheduling  Task: Generate a complete    │
│  system-level explanation and blog output.  Output Requirements: 1. Project Title 2. Scenario Overview 3.       │
│  Business Problem 4. Why Automation is Needed 5. Detailed Explanation of Each Agent (role, input, output,       │
│  responsibility) 6. End-to-End Workflow (step-by-step) 7. Input → Output mapping for each stage 8. Benefits of  │
│  this architecture 9. Risks / limitations 10. Future improvements and scalability  VERY IMPORTANT: Also         │
│  generate the following in diagram format:  1. Architecture Diagram (text/ASCII format) 2. Workflow Pipeline    │
│  Diagram (with arrows) 3. Agent Interaction Flow  Formatting: - Make the output visually structured and         │
│  presentation-ready - Use boxes, arrows, sections, and spacing - Keep it suitable for a final-year B.Tech       │
│  project report - Make it deep, not generic  Also include: - A full SEO-optimized blog post based on the        │
│  system - SEO title, meta description, tags, and slug - Publishing schedule suggestion  Tone: Professional,     │
│  structured, and technical with real-world clarity."                                                            │
│                                                                                                                 │
│  Focus areas:                                                                                                   │
│  - Latest EdTech innovations                                                                                    │
│  - AI in education                                                                                              │
│  - Smart classroom tools                                                                                        │
│  - Digital pedagogy trends                                                                                      │
│  - Adaptive learning systems                                                                                    │
│  - LMS/platform innovations                                                                                     │
│  - Gamified learning models                                                                                     │
│                                                                                                                 │
│  Audience context: schools                             

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Monitoring Agent                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Automated Learning Hub System for EdTech Company**                                                           │
│  ======================================================                                                         │
│                                                                                                                 │
│  ### Project Title                                                                                              │
│  Automated EdTech Learning Hub System                                                                           │
│                                                                                                                 │
│  ### Scenario Overview                                                                                          │
│  The EdTech company aims to automate the process of discovering, analyzing, writing, and publishing             │
│  educational content. The company wants to leverage multi-agent AI architecture to achieve this goal.           │
│                                                                                                                 │
│  ### Business Problem                                                                                           │
│  The manual effort involved in researching, summarizing, and writing educational content is time-consuming and  │
│  prone to errors. The company needs to reduce manual effort, ensure SEO optimization, and maintain consistent   │
│  publishing schedules.                                                                                          │
│                                                                                                                 │
│  ### Why Automation is Needed                                                                                   │
│  Automation is necessary to increase efficiency, reduce costs, and improve the quality of educational content.  │
│  By leveraging AI and machine learning, the company can analyze large amounts of data, identify trends, and     │
│  generate high-quality content.                                                                                 │
│                                                                                                                 │
│  ### Detailed Explanation of Each Agent                                                                         │
│                                                                                                                 │
│  #### 1. Trend Monitoring Agent                                                                                 │
│  * Role: Tracks latest EdTech innovations, AI in education, and digital learning trends                         │
│  * Input: Online research papers, articles, and social media posts                                              │
│  * Output: List of trending topics and keywords                                                                 │
│  * Responsibility: Identifies relevant trends and topics for the educational content                            │
│                                                                                                                 │
│  #### 2. Content Summarization Agent                                                                            │
│  * Role: Converts research into practical insights for 

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Take the final blog and package it for publication.                                                            │
│                                                                                                                 │
│  Generate:                                                                                                      │
│  1. Final polished article                                                                                      │
│  2. SEO title                                                                                                   │
│  3. Meta description                                                                                            │
│  4. Suggested slug                                                                                              │
│  5. Suggested tags                                                                                              │
│  6. Suggested weekly publishing schedule                                                                        │
│  7. Final clean ASCII diagrams                                                                                  │
│  8. A final section called "Deployment & Scaling Notes"                                                         │
│                                                                                                                 │
│  The final output must be highly structured and include:                                                        │
│  - Project Title                                                                                                │
│  - Scenario Overview                                                                                            │
│  - Business Problem                                                                                             │
│  - Why Automation is Needed                                                                                     │
│  - Agent Roles in Depth                                                                                         │
│  - End-to-End Workflow                                                                                          │
│  - Inputs and Outputs per Stage                                                                                 │
│  - Benefits                                                                                                     │
│  - Risks / Limitations                                                                                          │
│  - Future Scope                                                                                                 │
│  - ASCII Diagrams                                                                                               │
│                                                                                                                 │
│  Keep the formatting neat so it can be copied into a report, PPT, or documentation.                             │
│                                                                                                                 │
│  ID: feaf9e55-0776-426b-a6b3-6d932833322c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Publishing Agent                                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Take the final blog and package it for publication.                                                            │
│                                                                                                                 │
│  Generate:                                                                                                      │
│  1. Final polished article                                                                                      │
│  2. SEO title                                                                                                   │
│  3. Meta description                                                                                            │
│  4. Suggested slug                                                                                              │
│  5. Suggested tags                                                                                              │
│  6. Suggested weekly publishing schedule                                                                        │
│  7. Final clean ASCII diagrams                                                                                  │
│  8. A final section called "Deployment & Scaling Notes"                                                         │
│                                                                                                                 │
│  The final output must be highly structured and include:                                                        │
│  - Project Title                                                                                                │
│  - Scenario Overview                                                                                            │
│  - Business Problem                                                                                             │
│  - Why Automation is Needed                                                                                     │
│  - Agent Roles in Depth                                                                                         │
│  - End-to-End Workflow                                                                                          │
│  - Inputs and Outputs per Stage                                                                                 │
│  - Benefits                                                                                                     │
│  - Risks / Limitations                                                                                          │
│  - Future Scope                                                                                                 │
│  - ASCII Diagrams                                                                                               │
│                                                                                                                 │
│  Keep the formatting neat so it can be copied into a report, PPT, or documentation.                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Publishing Agent                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Automated EdTech Learning Hub System: Revolutionizing Education with AI-Powered Content Creation**           │
│  ====================================================================                                           │
│                                                                                                                 │
│  ## Project Title                                                                                               │
│  Automated EdTech Learning Hub System                                                                           │
│                                                                                                                 │
│  ## Scenario Overview                                                                                           │
│  The education sector is undergoing a significant transformation, driven by the advent of EdTech innovations,   │
│  AI-powered learning platforms, and digital learning trends. To stay ahead of the curve, schools and            │
│  educational institutions need to leverage these trends to provide personalized, efficient, and effective       │
│  learning experiences.                                                                                          │
│                                                                                                                 │
│  ## Business Problem                                                                                            │
│  The traditional process of content creation is time-consuming, labor-intensive, and often results in           │
│  low-quality content. Schools and educational institutions need a system that can automate the process of       │
│  content creation, providing high-quality educational content that is tailored to the needs of students and     │
│  teachers.                                                                                                      │
│                                                                                                                 │
│  ## Why Automation is Needed                                                                                    │
│  Automation is needed to improve the efficiency and effectiveness of the content creation process. The          │
│  Automated EdTech Learning Hub System can help schools provide personalized learning experiences for students,  │
│  improving outcomes and increasing engagement. The system can also help reduce the workload of teachers,        │
│  allowing them to focus on more critical tasks.                                                                 │
│                                                                                                                 │
│  ## Agent Roles in Depth                                                                                        │
│  The Automated EdTech Learning Hub System consists of four primary agents:                                      │
│                                                                                                                 │
│  1.  **Trend Monitoring Agent**: This agent is responsible for tracking the latest EdTech trends, innovations,  │
│  and research in the field of education. It analyzes industry reports, research papers, and social media to     │
│  identify emerging trends and technologies.            

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Take the final blog and package it for publication.                                                            │
│                                                                                                                 │
│  Generate:                                                                                                      │
│  1. Final polished article                                                                                      │
│  2. SEO title                                                                                                   │
│  3. Meta description                                                                                            │
│  4. Suggested slug                                                                                              │
│  5. Suggested tags                                                                                              │
│  6. Suggested weekly publishing schedule                                                                        │
│  7. Final clean ASCII diagrams                                                                                  │
│  8. A final section called "Deployment & Scaling Notes"                                                         │
│                                                                                                                 │
│  The final output must be highly structured and include:                                                        │
│  - Project Title                                                                                                │
│  - Scenario Overview                                                                                            │
│  - Business Problem                                                                                             │
│  - Why Automation is Needed                                                                                     │
│  - Agent Roles in Depth                                                                                         │
│  - End-to-End Workflow                                                                                          │
│  - Inputs and Outputs per Stage                                                                                 │
│  - Benefits                                                                                                     │
│  - Risks / Limitations                                                                                          │
│  - Future Scope                                                                                                 │
│  - ASCII Diagrams                                                                                               │
│                                                                                                                 │
│  Keep the formatting neat so it can be copied into a report, PPT, or documentation.                             │
│                                                                                                                 │
│  Agent: Publishing Agent                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────


✅ FINAL OUTPUT:

**Automated EdTech Learning Hub System: Revolutionizing Education with AI-Powered Content Creation**

## Project Title
Automated EdTech Learning Hub System

## Scenario Overview
The education sector is undergoing a significant transformation, driven by the advent of EdTech innovations, AI-powered learning platforms, and digital learning trends. To stay ahead of the curve, schools and educational institutions need to leverage these trends to provide personalized, efficient, and effective learning experiences.

## Business Problem
The traditional process of content creation is time-consuming, labor-intensive, and often results in low-quality content. Schools and educational institutions need a system that can automate the process of content creation, providing high-quality educational content that is tailored to the needs of students and teachers.

## Why Automation is Needed
Automation is needed to improve the efficiency and effectiveness of the content creation proces